In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# ============================================================
# LOAD TRAINING DATA
# ============================================================

csv_file = "train_dos.csv"

df = pd.read_csv(csv_file, index_col=0)

x_values = df.columns.astype(float).values
y_values = df.index.astype(float).values

matrix = df.values.astype(np.float32)

# ============================================================
# PREPARE DATASET
# ============================================================

X = []
Y = []

for i, y in enumerate(y_values):
    for j, x in enumerate(x_values):

        # Inputs
        X.append([y, x])

        # Output
        Y.append(matrix[i, j])

X = np.array(X, dtype=np.float32)
Y = np.array(Y, dtype=np.float32)

print("Total samples :", len(X))
print("Input shape   :", X.shape)
print("Output shape  :", Y.shape)

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

# ============================================================
# NORMALIZATION
# ============================================================

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled  = x_scaler.transform(X_test)

# Scale output also
y_train_scaled = y_scaler.fit_transform(
    y_train.reshape(-1, 1)
)

y_test_scaled = y_scaler.transform(
    y_test.reshape(-1, 1)
)

# ============================================================
# STRUCTURE OF THE NEURAL NETWORK
# ============================================================

model = Sequential([

    Dense(128, activation='relu', input_shape=(2,)),

    Dense(128, activation='relu'),

    Dense(64, activation='relu'),

    Dense(32, activation='relu'),

    Dense(1)

])

optimizer = Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=['mae']
)

model.summary()

# ============================================================
# EARLY STOPPING
# ============================================================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=30,
    restore_best_weights=True
)

# ============================================================
# TRAIN MODEL
# ============================================================

history = model.fit(

    X_train_scaled,
    y_train_scaled,

    validation_split=0.2,

    epochs=500,

    batch_size=64,

    callbacks=[early_stop],

    verbose=1
)

# ============================================================
# EVALUATE MODEL
# ============================================================

loss, mae = model.evaluate(
    X_test_scaled,
    y_test_scaled,
    verbose=0
)

print("\nScaled Test MAE:", mae)

# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_value(row_input, col_input):

    # Prepare input
    input_data = np.array(
        [[row_input, col_input]],
        dtype=np.float32
    )

    # Scale input
    input_scaled = x_scaler.transform(input_data)

    # Predict scaled output
    pred_scaled = model.predict(
        input_scaled,
        verbose=0
    )

    # Convert back to original scale
    prediction = y_scaler.inverse_transform(
        pred_scaled
    )

    return prediction[0][0]

C:\Users\tp662id\.conda\envs\mlenv\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
C:\Users\tp662id\.conda\envs\mlenv\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
C:\Users\tp662id\.conda\envs\mlenv\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
C:\Users\tp662id\.conda\envs\mlenv\lib\site-packages\tensorflow\python\frame

Total samples : 51850
Input shape   : (51850, 2)
Output shape  : (51850,)
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense (Dense)                (None, 128)               384       
_________________________________________________________________
dense_1 (Dense)              (None, 128)               16512     
_________________________________________________________________
dense_2 (Dense)              (None, 64)                8256      
_________________________________________________________________
dense_3 (Dense)              (None, 32)                2080      
_________________________________________________________________
dense_4 (Dense)              (None, 1)                 33        
Total params: 27,265
Trainable params: 27,265
Non-trainable pa

## To generate prediction (LDOS) map

In [2]:
import pandas as pd
import numpy as np
import csv
data_strain = pd.read_csv("raman_strain.csv", header=None)        # Input file name: "raman_strain.csv"

datastrain = np.array(data_strain, dtype=np.float32)
nrow, ncolumn = (datastrain.shape)

DataStrain = datastrain.flatten()

print("Total number of strain values: ", (DataStrain.shape))

csv_file = "train_dos.csv"

df = pd.read_csv(csv_file, index_col=0)

first_column = df.index.to_numpy(dtype=np.float32)
row = 1
column = 1

with open('dos_pred.csv', 'w', newline='') as file:               # Output file name: "dos_pred.csv"

    writer = csv.writer(file)
    writer.writerow(['' , ''] + y_values.tolist())
    # Loop over all strain values
    for j in range(len(DataStrain)):

        DOS = []   # reset list for every strain value

        strain_value = DataStrain[j]

        # Loop over energy axis
        for i in range(len(first_column)):

            pred = predict_value(
                first_column[i],
                strain_value
            )

            DOS.append(pred)

        writer.writerow([row, column] + DOS)
        row += 1
        if row == (nrow+1):
            row = 1
            column += 1
        print("Count of predicted DOS: ",j)
    DOS = []
    

Total number of strain values:  (1089,)
Count of predicted DOS:  0
Count of predicted DOS:  1
Count of predicted DOS:  2
Count of predicted DOS:  3
Count of predicted DOS:  4
Count of predicted DOS:  5
Count of predicted DOS:  6
Count of predicted DOS:  7
Count of predicted DOS:  8
Count of predicted DOS:  9
Count of predicted DOS:  10
Count of predicted DOS:  11
Count of predicted DOS:  12
Count of predicted DOS:  13
Count of predicted DOS:  14
Count of predicted DOS:  15
Count of predicted DOS:  16
Count of predicted DOS:  17
Count of predicted DOS:  18
Count of predicted DOS:  19
Count of predicted DOS:  20
Count of predicted DOS:  21
Count of predicted DOS:  22
Count of predicted DOS:  23
Count of predicted DOS:  24
Count of predicted DOS:  25
Count of predicted DOS:  26
Count of predicted DOS:  27
Count of predicted DOS:  28
Count of predicted DOS:  29
Count of predicted DOS:  30
Count of predicted DOS:  31
Count of predicted DOS:  32
Count of predicted DOS:  33
Count of predicted

## To test for a single strain value

In [14]:
import csv
csv_file = "train_dos.csv"
strain = 0.32                                                  # Input strain value
df = pd.read_csv(csv_file, index_col=0)
first_column = df.index.to_numpy()

with open('single_dos_pred.csv', 'w', newline='') as file:     # Output file name : "single_dos_pred.csv"
    writer = csv.writer(file)
    for i in range(len(first_column)):
        pred = predict_value(first_column[i], strain)
        writer.writerow([first_column[i], pred])